In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth+
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024  # Reduced from 2048 - speeds up generation significantly
 # Larger rank = smarter, but slower
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    offload_embedding = True, # Reduces VRAM by 1GB
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.2: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

...

Unsloth: Offloading embeddings to RAM to save 1.08 GB.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [3]:
lora_rank = 8

In [4]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("RUC-AIBOX/OlymMATH", "en-hard")

README.md: 0.00B [00:00, ?B/s]

OlymMATH-EN-HARD.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [5]:
import pandas as pd 
data = pd.DataFrame(ds['test'])
print(data.head())


                                             problem             answer  \
0  Let $a, b, c \in \mathbb{R}$, $a^3 b + b^3 c +...               2625   
1  If the distances from the eight vertices of a ...                210   
2  For $i = 1, 2, \cdots, n$, we have $x_i < 1$, ...                 11   
3  Find the minimum number of cubes (which can be...                  8   
4  Let $x$, $y$, $z$ be positive real numbers. Fi...  241 + 44\sqrt{30}   

    subject           unique_id  
0   Algebra  OlymMATH-HARD-0-EN  
1  Geometry  OlymMATH-HARD-1-EN  
2   Algebra  OlymMATH-HARD-2-EN  
3  Geometry  OlymMATH-HARD-3-EN  
4   Algebra  OlymMATH-HARD-4-EN  


In [ ]:
SYSTEM_PROMPT = """You are a math problem solver. Solve the problem step by step.
Finally, return only the final answer as an integer inside \\boxed{}."""

def format_prompt(question):
    return [
        {"role": "user", "content": f"{question}\n\nPlease reason step by step and put your final integer answer in \\boxed{{}}."}
    ]

In [7]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA adapters added successfully!")

Unsloth: Making `model.base_model.model.model` require gradients
LoRA adapters added successfully!


In [ ]:
# Prepare dataset - using first 2 problems for testing
import re

# Create training dataset from first 2 problems
train_data = []
for i in range(min(2, len(data))):
    row = data.iloc[i]
    question = row['problem']
    answer = row['answer']  # The ground truth answer
    
    train_data.append({
        "prompt": format_prompt(question),
        "answer": str(answer),
        "reasoning_effort": "low"
    })

print(f"Created dataset with {len(train_data)} problems")
print(f"Sample question: {train_data[0]['prompt'][0]['content'][:200]}...")
print(f"Sample answer: {train_data[0]['answer']}")

Created dataset with 2 problems
Sample question: Let $a, b, c \in \mathbb{R}$, $a^3 b + b^3 c + c^3 a = 3$, find the minimum value of the expression $f(a, b, c) = (\sum a^4)^4 + 1000 \sum a^2 b^2$.

Please reason step by step and put your final inte...
Sample answer: 2625


In [9]:
# Answer extraction function
def extract_boxed_answer(text):
    """Extract the answer from \\boxed{} in the model output"""
    # Try to find \boxed{...}
    patterns = [
        r'\\boxed\{([^{}]*)\}',  # Simple \boxed{answer}
        r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}',  # Nested braces
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        if matches:
            # Return the last match (final answer)
            answer_str = matches[-1].strip()
            try:
                # Try to parse as number
                # Handle fractions, negatives, etc.
                answer_str = answer_str.replace(',', '')  # Remove commas
                if '/' in answer_str:
                    # Handle fractions
                    parts = answer_str.split('/')
                    return float(parts[0]) / float(parts[1])
                return float(answer_str)
            except:
                return None
    return None

def parse_true_answer(answer_str):
    """Parse the ground truth answer to a number"""
    try:
        answer_str = str(answer_str).strip().replace(',', '')
        if '/' in answer_str:
            parts = answer_str.split('/')
            return float(parts[0]) / float(parts[1])
        return float(answer_str)
    except:
        return None

# Test extraction
test_text = "The answer is \\boxed{42}"
print(f"Test extraction: {extract_boxed_answer(test_text)}")

Test extraction: 42.0


In [10]:
# Reward Functions for RLVR

def format_reward(completions, **kwargs):
    """Reward for proper formatting with \\boxed{}"""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        # Check if response contains \boxed{}
        if "\\boxed{" in response:
            scores.append(0.5)  # Small bonus for correct format
        else:
            scores.append(-0.5)  # Penalty for missing format
    return scores

def answer_reward(completions, answer, **kwargs):
    """
    Main reward: negative distance from correct answer
    Reward = -1 * abs(TRUE - PREDICTED) / abs(TRUE) if TRUE != 0
    Reward = -1 * abs(PREDICTED) if TRUE == 0
    Reward = 0 if exactly correct
    """
    scores = []
    
    for completion, true_ans in zip(completions, answer):
        response = completion[0]["content"]
        predicted = extract_boxed_answer(response)
        true_value = parse_true_answer(true_ans)
        
        if predicted is None or true_value is None:
            # Can't parse answer - give penalty
            scores.append(-2.0)
            continue
        
        # Calculate distance-based reward
        if abs(predicted - true_value) < 1e-6:
            # Exactly correct!
            reward = 1.0  # Bonus for correct answer
        else:
            # Calculate relative error
            if abs(true_value) > 1e-6:
                relative_error = abs(true_value - predicted) / abs(true_value)
            else:
                relative_error = abs(predicted)
            
            # Negative reward based on error, capped at -2
            reward = -1.0 * min(relative_error, 2.0)
        
        scores.append(reward)
    
    return scores

# Test the reward function
test_completions = [[{"content": "The answer is \\boxed{42}"}]]
test_answers = ["42"]
print(f"Test reward (correct): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{40}"}]]
print(f"Test reward (close): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{0}"}]]
print(f"Test reward (far): {answer_reward(test_completions, test_answers)}")

Test reward (correct): [1.0]
Test reward (close): [-0.047619047619047616]
Test reward (far): [-1.0]


In [ ]:
# Create the HuggingFace Dataset
from datasets import Dataset

# Replicate the 2 problems to have enough data for 100 steps
# With batch_size=1 and 100 steps, we need at least 100 samples
replicated_data = train_data * 50  # 2 problems * 50 = 100 samples

dataset = Dataset.from_list(replicated_data)

# Calculate prompt length for configuration
sample_prompt = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize=False,
    add_generation_prompt=True,
    reasoning_effort="low"
)
max_prompt_length = len(tokenizer(sample_prompt)["input_ids"]) + 10  # Add buffer

print(f"Dataset size: {len(dataset)}")
print(f"Max prompt length: {max_prompt_length}")
print(f"Sample formatted prompt:\n{sample_prompt[:500]}...")

Dataset size: 2
Max prompt length: 178
Sample formatted prompt:
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-01-06

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Let $a, b, c \in \mathbb{R}$, $a^3 b + b^3 c + c^3 a = 3$, find the minimum value of the expression $f(a, b, c) = (\sum a^4)^4 + 1000 \sum a^2 b^2$.

Please ...


In [ ]:
# Configure GRPO Training
from trl import GRPOConfig, GRPOTrainer
import gc

# Cap completion length to something reasonable for speed
max_completion_length = min(max_seq_length - max_prompt_length, 512)  # Cap at 512 tokens

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 2,  # Number of completions per prompt
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 100,  # 100 steps for testing
    save_steps = 50,
    report_to = "none",
    output_dir = "outputs_grpo_test",
)

print(f"Max completion length: {max_completion_length}")
print("Training config ready!")

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2
Max completion length: 1870
Training config ready!


In [13]:
# Initialize the GRPO Trainer
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward,    # Reward for using \boxed{}
        answer_reward,    # Main reward: distance-based correctness
    ],
    args = training_args,
    train_dataset = dataset,
)

print("Trainer initialized!")

Trainer initialized!


In [ ]:
# Start training - 100 steps
# Monitor the 'reward' column in the output table - it should increase over time
import time

print("Starting training... (This will take a while - generation is the slow part)")
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time
print(f"\n{'='*50}")
print(f"Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Time per step: {training_time/100:.2f} seconds")
print(f"Estimated time for 1000 steps: {(training_time/100)*1000/60:.2f} minutes")

In [ ]:
# Test inference after training
text = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer

print("Testing trained model on first problem:")
print("="*50)
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)
print(f"\nExpected answer: {train_data[0]['answer']}")

In [ ]:
# Save the LoRA adapters
model.save_pretrained("gpt_oss_20b_math_lora_test")
tokenizer.save_pretrained("gpt_oss_20b_math_lora_test")
print("LoRA adapters saved to 'gpt_oss_20b_math_lora_test'")

## Optional: Save merged model or push to Hub

Uncomment the options below as needed:
- **LoRA only**: Smallest size, requires base model to load
- **Merged 16bit**: Full model in fp16
- **MXFP4**: GPT-OSS native precision, good for VLLM

In [ ]:
# Optional: Merge and save in different formats

# Save merged model in MXFP4 (GPT-OSS native precision)
# model.save_pretrained_merged("gpt_oss_20b_math_mxfp4", tokenizer, save_method="mxfp4")

# Save merged model in 16bit
# model.save_pretrained_merged("gpt_oss_20b_math_16bit", tokenizer, save_method="merged_16bit")

# Push to Hugging Face Hub (uncomment and add your token)
# model.push_to_hub_merged("your-username/gpt-oss-20b-math", tokenizer, token="hf_...", save_method="lora")